# Tiny LLM Lessons - Free Local Pipeline

Use this notebook in VS Code with the repo `.venv` selected as the kernel.

Setup from the repo root before opening the notebook:

```powershell
uv sync --python 3.12
uv add --dev ipykernel
```


## 1. Find The Repo Root And Use The Current Kernel

This notebook runs each script with the Python interpreter of the selected notebook kernel.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_dir = None

for candidate in candidates:
    if (candidate / "pyproject.toml").exists() and (candidate / "step9_train_tiny_transformer.py").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not find the repo root. Start VS Code from this project folder.")

os.chdir(project_dir)
print(f"project_dir: {project_dir}")
print(f"kernel python: {sys.executable}")

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, "-u", script, *args]
    print(">", shlex.join(cmd))
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


project_dir: C:\Users\kuchris\Desktop\github project\llm
kernel python: c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe


## 2. Check The Local Environment

This checks Python, PyTorch, and CUDA from the selected notebook kernel.


In [ ]:
import platform
import sys
import torch

print(f"python: {sys.version.split()[0]}")
print(f"platform: {platform.platform()}")
print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"cuda device: {torch.cuda.get_device_name(0)}")


python: 3.12.11
platform: Windows-11-10.0.26200-SP0
torch: 2.11.0+cu128
cuda available: True
cuda device: NVIDIA GeForce RTX 4080 Laptop GPU


## 3. Prepare Free Datasets


In [ ]:
run_py("download/step12_download_wikitext2.py")
run_py("step15_prepare_wikitext2.py")
run_py("step20_prepare_dolly.py")
run_py("step21_prepare_bea_grammar.py")
run_py("step22_prepare_free_tokenizer_train.py")


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u download/step12_download_wikitext2.py
saved: data\wikitext103.txt
characters: 541892487
preview: ' = Valkyria Chronicles III = \n\n Senj\u014d no Valkyria 3 : Unrecorded Chronicles ( Japanese : \u6226\u5834\u306e\u30f4\u30a1\u30eb\u30ad\u30e5\u30ea\u30a23 , lit . Valkyria of'
> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step15_prepare_wikitext2.py
input: data\wikitext103.txt
output: data\wikitext103_clean.txt
input chars: 541892487
output chars: 526197949
removed @-@ count: 885554
preview: '# Valkyria Chronicles III\n Senj\u014d no Valkyria 3: Unrecorded Chronicles ( Japanese: \u6226\u5834\u306e\u30f4\u30a1\u30eb\u30ad\u30e5\u30ea\u30a23, lit. Valkyria of the Battlefield 3 ), commonly referred to as Val'
> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step20_prepare_dolly.py
train examples: 14011
eval examples: 1000
saved train: data\dolly_train_eos_sft

## 4. Train Shared 6144 BPE Tokenizer


In [ ]:
run_py(
    "step13_bpe_tokenizer.py",
    "--vocab-size",
    "6144",
    "--max-chars",
    "10000000",
    "--special-token",
    "<unk>",
    "--special-token",
    "<eos>",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step13_bpe_tokenizer.py --vocab-size 6144 --max-chars 10000000 --special-token '<unk>' --special-token '<eos>'



saved tokenizer: tokenizers\free_bpe_6144.json
vocab size: 6144
sample: 'Europium is a chemical element.'
ids: [38, 1227, 1112, 363, 260, 4315, 511, 5381, 15]
decoded: 'Europium is a chemical element.'
round trip ok: True


## 5. Pretrain On WikiText-103

This trains general next-token prediction on clean WikiText-103.
If you stop training and want to continue later, rerun the same cell with `"--resume"` added.


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "free_wikitext103_pretrain_bpe",
    "--device",
    "auto",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset free_wikitext103_pretrain_bpe --device auto
device: cuda
step     0 / 20000 | loss 8.8932
step    20 / 20000 | loss 7.0874
step    40 / 20000 | loss 6.7729
step    60 / 20000 | loss 6.6858
step    80 / 20000 | loss 6.6781
step   100 / 20000 | loss 6.6849
step   120 / 20000 | loss 6.3949
step   140 / 20000 | loss 6.3122
step   160 / 20000 | loss 6.2658
step   180 / 20000 | loss 6.1370
step   200 / 20000 | loss 6.0370
autosaved checkpoint: checkpoints\free_wikitext103_pretrain_bpe\tiny_transformer.pt
step   220 / 20000 | loss 6.2219
step   240 / 20000 | loss 6.1941
step   260 / 20000 | loss 6.0199
step   280 / 20000 | loss 6.0433
step   300 / 20000 | loss 5.8399
step   320 / 20000 | loss 5.9275
step   340 / 20000 | loss 5.9260
step   360 / 20000 | loss 6.2034
step   380 / 20000 | loss 6.0374
step   400 / 20000 | loss 5.8157
autosaved checkpoint: checkpoints\free_wikitext10

## 6. General SFT On Dolly-15k

This loads the pretrain checkpoint and trains instruction/response behavior with response-only loss.


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "free_dolly_sft_bpe",
    "--device",
    "auto",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset free_dolly_sft_bpe --device auto
device: cuda
loaded init checkpoint: checkpoints\free_wikitext103_pretrain_bpe\tiny_transformer.pt
step     0 /  8000 | loss 6.6334
step    20 /  8000 | loss 6.7377
step    40 /  8000 | loss 5.6739
step    60 /  8000 | loss 5.3508
step    80 /  8000 | loss 5.9741
step   100 /  8000 | loss 5.6737
step   120 /  8000 | loss 5.5210
step   140 /  8000 | loss 5.6012
step   160 /  8000 | loss 4.9656
step   180 /  8000 | loss 4.7291
step   200 /  8000 | loss 4.9704
autosaved checkpoint: checkpoints\free_dolly_sft_bpe\tiny_transformer.pt
step   220 /  8000 | loss 5.2996
step   240 /  8000 | loss 4.9566
step   260 /  8000 | loss 5.1139
step   280 /  8000 | loss 5.4868
step   300 /  8000 | loss 5.0317
step   320 /  8000 | loss 5.0399
step   340 /  8000 | loss 4.6909
step   360 /  8000 | loss 5.2464
step   380 /  8000 | loss 5.3985
step   400 /  8000

## 7. Grammar SFT On BEA


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "free_bea_grammar_sft_bpe",
    "--device",
    "auto",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step9_train_tiny_transformer.py --preset free_bea_grammar_sft_bpe --device auto
device: cuda
loaded init checkpoint: checkpoints\free_dolly_sft_bpe\tiny_transformer.pt
step     0 /  4000 | loss 4.5046
step    20 /  4000 | loss 4.1960
step    40 /  4000 | loss 4.4793
step    60 /  4000 | loss 4.3673
step    80 /  4000 | loss 3.9484
step   100 /  4000 | loss 3.9614
step   120 /  4000 | loss 3.7560
step   140 /  4000 | loss 4.1277
step   160 /  4000 | loss 4.7326
step   180 /  4000 | loss 4.5157
step   200 /  4000 | loss 4.7386
autosaved checkpoint: checkpoints\free_bea_grammar_sft_bpe\tiny_transformer.pt
step   220 /  4000 | loss 4.0553
step   240 /  4000 | loss 3.8201
step   260 /  4000 | loss 4.3226
step   280 /  4000 | loss 4.1747
step   300 /  4000 | loss 3.9995
step   320 /  4000 | loss 4.5214
step   340 /  4000 | loss 4.2409
step   360 /  4000 | loss 4.1337
step   380 /  4000 | loss 3.9307
step   400 /  400

## 8. Generate From The Final Model


In [ ]:
run_py(
    "step10_generate.py",
    "--preset",
    "free_bea_grammar_sft_bpe",
    "--device",
    "auto",
    "--temperature",
    "0.1",
    "--max-new-tokens",
    "40",
    "--prompt",
    "Correct the grammar, spelling, and punctuation mistakes in the following text.\n\n### Input:\nShe go to school yesterday.",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step10_generate.py --preset free_bea_grammar_sft_bpe --device auto --temperature 0.1 --max-new-tokens 40 --prompt 'Correct the grammar, spelling, and punctuation mistakes in the following text.

### Input:
She go to school yesterday.'
device: cuda
### Instruction:
Correct the grammar, spelling, and punctuation mistakes in the following text.

### Input:
She go to school yesterday.

### Response:
The best places are a lot of people who are in the world.

In the other hand, I think that I have a lot of people who are in the world.




In [ ]:
run_py(
    "step10_generate.py",
    "--preset",
    "free_dolly_sft_bpe",
    "--device",
    "auto",
    "--temperature",
    "0.3",
    "--max-new-tokens",
    "40",
    "--prompt",
    "write me a story about a cat who loves to code.",
)


> 'c:\Users\kuchris\Desktop\github project\llm\.venv\Scripts\python.exe' -u step10_generate.py --preset free_dolly_sft_bpe --device auto --temperature 0.3 --max-new-tokens 40 --prompt 'write me a story about a cat who loves to code.'
device: cuda
### Instruction:
write me a story about a cat who loves to code.

### Response:
These are a great way to learn a good life.


## 9. Optional: Resume A Stage

Change the preset to whichever stage you want to continue. `--max-iters` is the total target step count, not extra steps.


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "free_wikitext103_pretrain_bpe",
    "--device",
    "auto",
    "--max-iters",
    "20000",
    "--resume",
)


## 10. Optional: Short Smoke Test

Use this if you only want to test that training starts on your local machine.


In [ ]:
run_py(
    "step9_train_tiny_transformer.py",
    "--preset",
    "free_wikitext103_pretrain_bpe",
    "--device",
    "auto",
    "--max-iters",
    "2",
    "--checkpoint",
    "checkpoints/local_smoke_test.pt",
)
